# Nuclear Shape Analysis

**Author: Fabien Bertillot**  

---
## 1. Objective

Compute nuclear shape descriptors from labeled nuclei segmentation masks of colorectal cancer tumor microarray (TMA) samples.

## 2. Dependencies

- Libraries: `cupy`, `imageio`, `skimage`, `scipy`, `spatial_efd`
- Files required:  
  - `mask.png` — Labeled mask image where each nucleus has a unique integer label (PNG format)
---

## 3. Notebook Roadmap

1. **Load Dependencies**  
2. **Define Nuclear Shape Feature Extraction Functions**  
12. **Batch Processing**

---

## 1) Load Dependencies

In [1]:
import cupy as cp
import numpy as np
import pandas as pd
import imageio.v3 as imageio
from scipy.ndimage import center_of_mass
import cv2
import time
import glob
import pathlib
from collections import defaultdict
from skimage.segmentation import expand_labels
from scipy.spatial import ConvexHull
import matplotlib.path as mpath
from typing import List, Optional, Tuple, Dict
import spatial_efd
import os 
num_devices = cp.cuda.runtime.getDeviceCount()
print(f"Number of CUDA devices: {num_devices}")

for i in range(num_devices):
    props = cp.cuda.runtime.getDeviceProperties(i)
    print(f"Device {i}: {props['name']}")

Number of CUDA devices: 1
Device 0: b'NVIDIA RTX A2000 Laptop GPU'


In [ ]:
from watermark import watermark
%reload_ext watermark
%watermark --iversions

## 2) Nuclear Shape Feature Extraction Functions

In [5]:
def extract_contours(
    mask: np.ndarray,
    tile_size: int = 1000,
    overlap: int = 100,
    chunk_size: int = 100,
    merge_disconnected: bool = True
) -> Dict[int, List[List[Tuple[int, int]]]]:
    
    """
    Extracts contours from a labeled mask using a tile-based approach, 
    enabling efficient processing of extremely large masks.

    This method divides the mask into overlapping tiles to minimize 
    edge artifacts and processes label contours in chunks to optimize memory usage. 
    It uses GPU acceleration for efficiency and includes optional merging of 
    disconnected regions belonging to the same label.

    Parameters
    ----------
    mask : np.ndarray
        A 2D array where different integer values represent different objects.
    tile_size : int, optional, default=1000
        The size of each processing tile in pixels.
    overlap : int, optional, default=100
        The overlap between tiles in pixels to avoid discontinuities at edges.
    chunk_size : int, optional, default=100
        The number of labels to process in each chunk (for memory optimization).
    merge_disconnected : bool, optional, default=True
        If True, merges disconnected regions with the same label into a single entry.

    Returns
    -------
    Dict[int, List[List[Tuple[int, int]]]]
        A dictionary mapping each label to a list of contours.
        - If `merge_disconnected` is True, each label will have a single list 
          containing all contour points.
        - If False, each label may have multiple separate contours.
    """
    
    # Get mask dimensions
    height, width = mask.shape
    
    # Dictionary to store contours for each label
    all_contours = defaultdict(list)
    
    # Get unique labels excluding background (0)
    unique_labels = np.unique(mask)
    unique_labels = unique_labels[unique_labels != 0]
    
    # Calculate number of tiles in each dimension
    n_tiles_h = max(1, (height + tile_size - 1) // tile_size)
    n_tiles_w = max(1, (width + tile_size - 1) // tile_size)
    
    # Store contours with their bounding boxes to help with deduplication
    contour_info = defaultdict(list)
    
    # Process each tile with generous overlap
    for i in range(n_tiles_h):
        for j in range(n_tiles_w):
            # Calculate core tile coordinates
            core_start_h = i * tile_size
            core_start_w = j * tile_size
            core_end_h = min(core_start_h + tile_size, height)
            core_end_w = min(core_start_w + tile_size, width)
            
            # Calculate expanded tile coordinates with overlap
            tile_start_h = max(0, core_start_h - overlap)
            tile_start_w = max(0, core_start_w - overlap)
            tile_end_h = min(height, core_end_h + overlap)
            tile_end_w = min(width, core_end_w + overlap)
            
            # Extract expanded tile
            tile = mask[tile_start_h:tile_end_h, tile_start_w:tile_end_w]
            
            # Find labels in this tile
            tile_labels = np.unique(tile)
            tile_labels = tile_labels[tile_labels != 0]
            
            # Skip if tile has no objects
            if len(tile_labels) == 0:
                continue
            
            # Process tile with GPU (in chunks to manage memory)
            for k in range(0, len(tile_labels), chunk_size):
                chunk_labels = tile_labels[k:k+chunk_size]
                
                # Transfer tile to GPU
                tile_gpu = cp.asarray(tile)
                
                for label in chunk_labels:
                    # Create binary mask for current label
                    binary_mask_gpu = (tile_gpu == label).astype(cp.uint8)
                    binary_mask = cp.asnumpy(binary_mask_gpu)
                    
                    # Find contours
                    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    # Skip if no contours found
                    if not contours:
                        continue
                    
                    # Process each contour
                    for contour in contours:
                        # Adjust coordinates to global mask coordinates
                        adjusted_contour = contour.copy()
                        adjusted_contour[:, :, 0] += tile_start_w
                        adjusted_contour[:, :, 1] += tile_start_h
                        
                        # Get a bounding box for this contour (x, y, w, h)
                        x, y, w, h = cv2.boundingRect(adjusted_contour)
                        
                        # Check if this contour is completely within the core tile area
                        # or if it extends beyond and should be handled in another tile
                        contour_in_core = (
                            x >= core_start_w and x + w <= core_end_w and
                            y >= core_start_h and y + h <= core_end_h
                        )
                        
                        contour_crosses_border = (
                            (x < tile_start_w + overlap or x + w > tile_end_w - overlap) or
                            (y < tile_start_h + overlap or y + h > tile_end_h - overlap)
                        )
                        
                        # Store contour for this label with its bounding box for deduplication
                        contour_points = adjusted_contour.reshape(-1, 2).tolist()
                        contour_info[int(label)].append({
                            'points': contour_points,
                            'bbox': (x, y, w, h),
                            'in_core': contour_in_core,
                            'crosses_border': contour_crosses_border,
                            'area': cv2.contourArea(adjusted_contour)
                        })
                
                # Free GPU memory
                del tile_gpu
                cp.get_default_memory_pool().free_all_blocks()
    
    # Post-process to deduplicate contours
    for label, contours in contour_info.items():
        # Sort contours by area (largest first) and whether they're in a core region
        sorted_contours = sorted(
            contours, 
            key=lambda c: (-c['in_core'], -c['area'])
        )
        
        # Keep track of which contours to keep
        kept_contours = []
        kept_bboxes = []
        
        # Process each contour
        for contour in sorted_contours:
            x, y, w, h = contour['bbox']
            
            # Calculate overlap with existing contours
            is_duplicate = False
            for kept_bbox in kept_bboxes:
                kx, ky, kw, kh = kept_bbox
                
                # Check if bounding boxes overlap significantly
                overlap_x = max(0, min(x + w, kx + kw) - max(x, kx))
                overlap_y = max(0, min(y + h, ky + kh) - max(y, ky))
                overlap_area = overlap_x * overlap_y
                contour_area = w * h
                
                # If overlap is significant, consider it a duplicate
                if overlap_area > 0.8 * contour_area:
                    is_duplicate = True
                    break
            
            # If not a duplicate, keep this contour
            if not is_duplicate:
                kept_contours.append(contour['points'])
                kept_bboxes.append(contour['bbox'])
        
        # Store deduplicated contours
        if merge_disconnected and kept_contours:
            # If merging disconnected regions, combine all contours for this label
            all_points = []
            for contour in kept_contours:
                all_points.extend(contour)
            all_contours[label] = [all_points]
        else:
            # Otherwise, keep each contour separate
            all_contours[label] = kept_contours
    
    return dict(all_contours)

def compute_cell_orientation(coordinates: np.ndarray) -> Tuple[float, float]:
    """
    Computes the primary orientation of a cell based on its coordinates.

    The function centers the coordinates, calculates the covariance matrix, 
    and determines the eigenvector corresponding to the smallest eigenvalue, 
    which represents the orientation of the shape.

    Parameters
    ----------
    coordinates : np.ndarray
        A NumPy array of shape (N, 2) containing (x, y) coordinates of the cell boundary.

    Returns
    -------
    Tuple[float, float]
        A tuple representing the orientation vector (x, y).
    """
    
    coordinates -= np.mean(coordinates, axis=0)  # Centered at 0
    covariance = np.dot(coordinates.T.copy(), coordinates)
    eigenValues, eigenVectors = np.linalg.eig(covariance)  
    
    orientation_vector = eigenVectors[np.argmin(eigenValues),:]
    
    return orientation_vector

def calculate_polygon_perimeter(vertices: List[Tuple[float, float]]) -> float:
    """
    Computes the perimeter of a polygon given its vertices using an optimized 
    8-connected distance formula.

    Parameters
    ----------
    vertices : List[Tuple[float, float]]
        A list of (x, y) coordinate tuples representing the polygon vertices.

    Returns
    -------
    float
        The perimeter of the polygon.
    """

    # Convert to numpy array for faster operations
    points = np.asarray(vertices)
    
    # Calculate differences between consecutive points using vectorized operations
    # Roll the array to get pairs of consecutive points
    rolled_points = np.roll(points, -1, axis=0)
    diffs = rolled_points - points
    
    # Calculate segment lengths using vectorized operations
    dx = np.abs(diffs[:, 0])
    dy = np.abs(diffs[:, 1])
    
    # Apply 8-connected formula vectorized
    lengths = np.maximum(dx, dy) + 0.41421356 * np.minimum(dx, dy)
    
    # Sum up all lengths
    return np.sum(lengths)

def calculate_polygon_axes_accurate(vertices: List[Tuple[float, float]]) -> Tuple[float, float]:
    """
    Computes the major and minor axes of a polygon using a grid-sampling 
    approach for increased accuracy.

    This method:
    - Determines the polygon's bounding box.
    - Samples a grid of points within the bounding box.
    - Identifies points inside the polygon.
    - Computes the covariance matrix to estimate the major and minor axes.

    Parameters
    ----------
    vertices : List[Tuple[float, float]]
        A list of (x, y) coordinate tuples representing the polygon vertices.

    Returns
    -------
    Tuple[float, float]
        A tuple containing the estimated major axis and minor axis lengths.
    """

    # Convert to numpy array
    points = np.asarray(vertices)
    
    # Get bounding box
    min_x = np.min(points[:, 0])
    max_x = np.max(points[:, 0])
    min_y = np.min(points[:, 1])
    max_y = np.max(points[:, 1])
    
    # Create polygon path once
    polygon_path = mpath.Path(points)
    
    # Calculate what grid size to use based on polygon size
    # For reasonable accuracy without excessive computation
    grid_size = int(min(max(20, len(vertices) * 2), 100))
    
    # Generate grid of points more efficiently
    x_range = np.linspace(min_x, max_x, grid_size)
    y_range = np.linspace(min_y, max_y, grid_size)
    grid_x, grid_y = np.meshgrid(x_range, y_range)
    grid_points = np.stack((grid_x.flatten(), grid_y.flatten()), axis=1)
    
    # Check which points are inside the polygon
    is_inside = polygon_path.contains_points(grid_points)
    inside_points = grid_points[is_inside]
    
    # Fallback if not enough points
    if len(inside_points) < 3:
        # Use original vertices as fallback
        inside_points = points
    
    # Calculate centroid
    centroid = np.mean(inside_points, axis=0)
    
    # Calculate covariance matrix (using [y, x] order as in PyRadiomics)
    centered = inside_points - centroid
    cov_matrix = np.zeros((2, 2))
    cov_matrix[0, 0] = np.mean(centered[:, 1]**2)  # y variance
    cov_matrix[1, 1] = np.mean(centered[:, 0]**2)  # x variance
    cov_matrix[0, 1] = cov_matrix[1, 0] = np.mean(centered[:, 1] * centered[:, 0])  # xy covariance
    
    # Calculate eigenvalues
    eigenvalues, _ = np.linalg.eigh(cov_matrix)
    
    # Sort eigenvalues (eigh returns them in ascending order)
    eigenvalues = eigenvalues[::-1]
    
    # Calculate axis lengths with the factor of 4 as in PyRadiomics
    return 4.0 * np.sqrt(eigenvalues[0]), 4.0 * np.sqrt(eigenvalues[1])

def compute_polygon_area(vertices: List[Tuple[float, float]]) -> float:
    """
    Computes the area of a polygon using the shoelace formula.

    This method efficiently calculates the area of a simple closed polygon 
    using a vectorized approach.

    Parameters
    ----------
    vertices : List[Tuple[float, float]]
        A list of (x, y) coordinate tuples representing the polygon vertices.

    Returns
    -------
    float
        The computed area of the polygon.
    """
    
    # Convert to numpy array
    points = np.asarray(vertices)
    
    # Vectorized calculation
    x = points[:, 0]
    y = points[:, 1]
    
    # Shoelace formula
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def compute_shape_features(
    polygon_tuple: List[Tuple[float, float]], 
    harmonics: int = 10
) -> Dict[str, float]:
    """
    Computes shape features directly from polygon vertices, optimizing performance 
    while prioritizing accuracy for key metrics such as major and minor axes.

    This function extracts geometric and morphological features, including:
    - Area, perimeter, convex area, and aspect ratio.
    - Compactness, circularity, and solidity.
    - Elliptic Fourier Descriptors (EFD) for shape analysis.
    - Derived shape metrics such as eccentricity and roundness.

    Parameters
    ----------
    polygon_tuple : List[Tuple[float, float]]
        A list of (x, y) coordinate tuples defining the polygon shape.
    harmonics : int, optional, default=10
        The number of harmonics used in the Elliptic Fourier Descriptors (EFD) computation.

    Returns
    -------
    Dict[str, float]
        A dictionary containing shape features with the following keys:
        - 'major_axis': Length of the major axis.
        - 'minor_axis': Length of the minor axis.
        - 'perimeter': Perimeter of the polygon.
        - 'convex_area': Area of the convex hull.
        - 'area': Area of the polygon.
        - 'circularity': Measure of how close the shape is to a perfect circle.
        - 'roundness': Ratio related to how round the shape is.
        - 'solidity': Ratio of the area to the convex area.
        - 'aspect_ratio': Ratio of the major axis to the minor axis.
        - 'compactness': Measure of shape compactness.
        - 'efc_ratio': Elliptic Fourier Coefficients ratio, quantifying shape variation.
        - Additional EFD coefficients in the format 'efd_a{i}b{j}'.

    Notes
    -----
    - The function first computes fundamental geometric properties.
    - The convex hull is computed using `scipy.spatial.ConvexHull`.
    - If the convex hull computation fails (e.g., for degenerate cases), the convex area falls back to the original area.
    - Elliptic Fourier Descriptors (EFD) are computed and normalized for shape comparison.
    - The `efc_ratio` is derived from the first harmonic coefficients of the EFD series.
    """
    
    # Calculate area with optimized method
    area = compute_polygon_area(polygon_tuple)
    
    # Calculate perimeter with optimized method
    perimeter = calculate_polygon_perimeter(polygon_tuple)
    
    # Calculate axes with more accurate (but still optimized) method
    major_axis, minor_axis = calculate_polygon_axes_accurate(polygon_tuple)
    
    # Compute convex hull and its area
    points = np.asarray(polygon_tuple)
    try:
        hull = ConvexHull(points)
        convex_area = hull.volume
    except:
        # Fallback for degenerate cases
        convex_area = area
    
    # Calculate derived metrics using fast operations
    solidity = area / convex_area if convex_area > 0 else 1.0
    compactness = perimeter**2 / (4*np.pi*area) if area > 0 else float('inf')
    aspect_ratio = major_axis / minor_axis if minor_axis > 0 else float('inf')
    circularity = 4*np.pi*(area/perimeter**2) if perimeter > 0 else 0.0
    roundness = 4*(area/(np.pi*major_axis**2)) if major_axis > 0 else 0.0

    # EFD coefficients and EFC ratio 
    x_polygon = [point[0] for point in polygon_tuple]
    y_polygon = [point[1] for point in polygon_tuple]
    
    EFDcoeffs = spatial_efd.CalculateEFD(x_polygon, y_polygon, harmonics)
    EFDcoeffs, rotation = spatial_efd.normalize_efd(EFDcoeffs, size_invariant = True)
    
    MinorAxisArray = []
    MajorAxisArray = []
    
    for i in range(len(EFDcoeffs)):
        MinorAxis = (EFDcoeffs[i][0]**2 + EFDcoeffs[i][2]**2)**(1/2)
        MajorAxis = (EFDcoeffs[i][1]**2 + EFDcoeffs[i][3]**2)**(1/2)
        MinorAxisArray.append(MinorAxis)
        MajorAxisArray.append(MajorAxis)          
    efc_ratio = (MinorAxisArray[0] + MajorAxisArray[0])/( sum(MinorAxisArray[1:]) + sum(MajorAxisArray[1:]))
    
    # dict_efd = {}
    # for i in range(harmonics):
    #     dict_efd.update({f'efd_a{i}b{j}': EFDcoeffs[i,j] for j in range(4)})
    
    # Build the shape features dictionary
    shape_features = {
        'major_axis': major_axis,
        'minor_axis': minor_axis,
        'perimeter': perimeter,
        'convex_area': convex_area,
        'area': area,
        'circularity': circularity,
        'roundness': roundness,
        'solidity': solidity,
        'aspect_ratio': aspect_ratio,
        'compactness': compactness,
        'efc_ratio': efc_ratio
    }

    # shape_features.update(dict_efd)
    
    return shape_features

def compute_features_multilabel(
    mask_path: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    """
    Computes first-order statistics for multiple labels in a single-pass per channel.

    This function processes all labels simultaneously, followed by a CPU-based finalization
    for each label. This approach can be more efficient than gathering statistics label by label,
    especially when dealing with a large number of labels, as it requires only one pass per channel.

    Parameters
    ----------
    image_path : str
        Path to a TIFF image with shape (N, X, Y), where N is the number of channels.
    mask_path : str
        Path to a segmentation mask image with shape (X, Y), containing integer labels.

    Returns
    -------
    shape_features_df : pd.DataFrame
        A DataFrame containing shape-based features for each label.
    """

    # --- 1) Load mask

    mask_np  = imageio.imread(mask_path)   # shape (X, Y)

    # --- 2) Shape features
    contours = extract_contours(mask_np, tile_size=200, overlap=20, chunk_size=50)

    all_shape_features = []
    for label in list(contours.keys()):
        coords = contours[label]
        polygon = [(pts[0], pts[1]) for pts in coords[0]]
        
        # Add label to shape features dictionary
        shape_features = compute_shape_features(polygon)
        shape_features['label'] = label

        coordinates = np.array([[pts[0], pts[1]] for pts in coords[0]]).astype('float64')
        orientation_vector = compute_cell_orientation(coordinates)
        shape_features['orientation_x'] = orientation_vector[0]
        shape_features['orientation_y'] = orientation_vector[1]
        
        # Append to list of features
        all_shape_features.append(shape_features)

    # Create DataFrame from the list of feature dictionaries
    shape_features_df = pd.DataFrame(all_shape_features
    
    return shape_features_df
else:
    return False

## 3) Batch Processing

In [ ]:
MASK_DIR = r''
SHAPE_FEATURE_DIR = r''

mask_paths = glob.glob(MASK_DIR + r'/*.tif')

for mask_path in mask_paths:
    start = time.time()
    image_name = pathlib.PurePath(mask_path).name

    output_shape_path = f'{SHAPE_FEATURE_DIR}/features_image={image_name.replace(".png", ".h5")}'

    if os.path.exists(output_shape_path):
        print(f"features already computed for {image_name}.")
        continue

    result = compute_features_multilabel(mask_path)

    if result is not False:
        result['Image_Name']= image_name.replace('.tif','')
        result.to_hdf(output_shape_path, key='df', mode='w')
        end = time.time()
        print(f"Processed {image_name} in {end - start:.2f} seconds.")

        